# 01 — Prediction horizon vs predictor count

Replaces tzhou's `260419_MSEvis.ipynb` and `260426_uvvis.ipynb`. Same plots,
but the notebook is driven entirely by `latentsky.registry`:
no hardcoded checkpoint lists, no filename parsing.

Pattern:
1. `list_runs()` — discover every trained experiment.
2. Group runs by predictor set (the `input_vars` field).
3. For each (predictor set, lead time), evaluate on `test_piece_ids` and collect metrics.
4. Plot metric vs lead time, one curve per predictor set.

In [ ]:
import sys, pathlib
REPO = pathlib.Path('..').resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from latentsky import list_runs, load_run

names = list_runs()
print(f'{len(names)} runs available')
for n in names:
    r = load_run(n)
    bv = f'{r.best_val_loss:.4f}' if r.best_val_loss is not None else 'imported'
    print(f'  {n:30s}  lead={r.lead_hours:>4.2f}h  vars={len(r.predictors)}  best_val={bv}')

## Group runs by predictor set

Each unique `tuple(input_vars)` becomes one curve. Lead time is read from the
config — no parsing of `lead0.25h_ep155` filenames anywhere.

In [2]:
from collections import defaultdict

# Identify each predictor set with a short label.
def predictor_label(input_vars):
    extras = [v for v in input_vars if v not in ('precipw', 't2m', 'olrtoa')]
    if not extras:
        return '3var (pw,t2m,olr)'
    if any('mse' in v for v in extras) and any('uzonal' in v for v in extras):
        return '9var (mse+uv)'
    if any('mse' in v for v in extras):
        return '5var (mse)'
    if any('relhum' in v for v in extras):
        return '5var (relhum)'
    return ','.join(extras)

groups = defaultdict(list)
for n in list_runs():
    r = load_run(n)
    groups[predictor_label(r.predictors)].append(r)

for k, runs in groups.items():
    runs.sort(key=lambda r: r.lead_hours)
    print(f'{k}: {[r.lead_hours for r in runs]}')

5var (relhum): [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
5var (mse): [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0]
9var (mse+uv): [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 4.0, 6.0]


## Load metrics

Inference is done by `scripts/eval_horizon_vs_predictors.py` (submitted via
`submit/eval_horizon.pbs`), which writes one row per run to
`docs/horizon_vs_predictors.csv`. This notebook just loads it.

In [ ]:
df = pd.read_csv(REPO / 'docs' / 'horizon_vs_predictors.csv')
df = df.sort_values(['predictor_set', 'lead_h']).reset_index(drop=True)
print(f'{len(df)} rows, {df["predictor_set"].nunique()} predictor sets')
df

## Plot: each metric vs lead time, one curve per predictor set

In [ ]:
metric_cols = ['R2_all', 'R2_rain', 'MSE', 'MAE', 'Correlation', 'JS_div']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
for ax, m in zip(axes.flat, metric_cols):
    for label, sub in df.groupby('predictor_set'):
        ax.plot(sub['lead_h'], sub[m], marker='o', label=label)
    ax.set_xlabel('lead time (h)')
    ax.set_ylabel(m)
    ax.set_title(m)
    if m.startswith('R2'):
        ax.set_ylim(0, 1)
    ax.grid(True, alpha=0.3)
axes[0, 0].legend(fontsize=8, loc='best')
fig.suptitle('Skill vs forecast horizon, by predictor set', fontsize=13)
fig.tight_layout()
plt.show()